# Chapter 14 — Stress
### *Modern Strain Gage Technology* — Companion Notebook

**Stress is the internal counterpart to external load** — and unlike the force you apply from the outside, its value at a point depends entirely on the orientation of the plane you choose to examine.

Companion notebook to Chapter 14 — Stress.

When a force is applied to a body, the material responds by developing internal forces distributed across every cross-section, and the intensity of those internal forces, expressed per unit area, is stress. But stress at a point is not a single scalar: rotate the plane you cut through the material, and both the magnitude and direction of the *traction* acting on it change.

This notebook demonstrates *Cauchy's formula*: given a stress state and the unit normal of a plane, the traction vector on that plane is simply the stress tensor dotted with the normal, $\mathbf{t} = \boldsymbol{\sigma}\cdot\mathbf{n}$. Sweeping the plane orientation through all angles in 2D produces the polar diagram below — a direct visualization of how stress varies with plane orientation, and where the extremes (the principal stresses and the maximum shear) occur.

**This notebook demonstrates:**
1. Traction on arbitrary planes via Cauchy's formula — a polar plot of normal and shear traction versus plane orientation, with principal-stress extraction.

In [ ]:
# Colab setup — uncomment to install dependencies in a fresh environment:
# !pip install numpy matplotlib

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11, 'axes.titlesize': 12})

# All figures for this chapter are written here (path preserved for the book build).
OUT_DIR = Path("../src/media/ch14")
OUT_DIR.mkdir(parents=True, exist_ok=True)

---
## 1 — Cauchy's Formula: Traction on an Arbitrary Plane

Cauchy's stress theorem states that the traction vector **t** on any plane with unit normal **n** is given by **t** = **σ** · **n**, where **σ** is the stress tensor at that point. This is a linear relationship: double the stress, double the traction. Rotate the plane, and both the normal component (σₙ = **t** · **n**) and the shear component (τₛ = |**t** − σₙ**n**|) change in a way governed entirely by the tensor.

The polar plot makes this rotation-dependence visible. The radius at each angle gives the magnitude of the traction component in that plane orientation; because a polar radius cannot be negative, the normal component is plotted as a magnitude and its sign is shown by shading (tinted for tension, dotted for compression). The angle at which the normal traction is maximum (and shear is zero) defines the first principal direction — the plane on which a gage oriented along the normal would read the maximum possible strain for this stress state. The angle of maximum shear is 45° from the principal directions, which is why torsion gages are always mounted at ±45° to the shaft axis.

In [ ]:
# Cauchy's formula: the traction on a plane is t = sigma . n.
# Given a plane's unit normal n, we resolve that traction into a component
# normal to the plane (sigma_n) and a component parallel to it (tau_s), then
# sweep the plane orientation through a full turn to see how both vary.

# --- Representative biaxial (plane-stress) state; all values in MPa ---
SIGMA_X = 80.0    # normal stress on the x-face  [MPa]
SIGMA_Y = -20.0   # normal stress on the y-face  [MPa]
TAU_XY = 30.0     # in-plane shear stress        [MPa]

N_ANGLES = 1000   # angular resolution of the orientation sweep
SAVE_DPI = 150    # export resolution for the saved figure

sigma = np.array([[SIGMA_X, TAU_XY],
                  [TAU_XY,  SIGMA_Y]])


def principal_stresses(stress_tensor):
    """Return (sigma_1, sigma_2, theta_1) for a 2x2 symmetric stress tensor.

    sigma_1 >= sigma_2 are the principal (normal) stresses; theta_1 is the
    orientation of the first principal direction in radians, measured from the
    x-axis. Uses eigh, which is exact for the symmetric Cauchy stress tensor.
    """
    eigvals, eigvecs = np.linalg.eigh(stress_tensor)  # eigh returns ascending order
    sigma_2, sigma_1 = eigvals
    v1 = eigvecs[:, 1]                                 # eigenvector of the larger eigenvalue
    theta_1 = np.arctan2(v1[1], v1[0])
    return sigma_1, sigma_2, theta_1


def traction_components(stress_tensor, theta):
    """Normal and shear traction on planes whose normals sweep through `theta`.

    For each angle in `theta` (radians) the plane's unit normal is
    n = (cos t, sin t). Cauchy's formula gives the traction t = sigma . n, which
    we split into the normal component sigma_n = t . n and the shear magnitude
    tau_s = |t - sigma_n n|. Returns arrays (sigma_n, tau_s) aligned with `theta`.
    """
    normals = np.stack([np.cos(theta), np.sin(theta)])   # shape (2, N)
    tractions = stress_tensor @ normals                  # shape (2, N)
    sigma_n = np.sum(tractions * normals, axis=0)        # t . n
    shear_vec = tractions - sigma_n * normals
    tau_s = np.linalg.norm(shear_vec, axis=0)
    return sigma_n, tau_s


p1_val, p2_val, theta_p1 = principal_stresses(sigma)
max_ts = (p1_val - p2_val) / 2          # maximum in-plane shear = radius of the tensor

theta_rad = np.radians(np.linspace(0, 360, N_ANGLES))
tn_vals, ts_vals = traction_components(sigma, theta_rad)

fig = plt.figure(figsize=(9, 8))
ax = fig.add_subplot(111, projection='polar')

# A polar radius cannot be negative, so plot the MAGNITUDE of the normal traction
# on a zero-origin radius and carry its sign by shading: tension lobes are tinted,
# compression lobes are dotted. The two curves are told apart by line style as
# well as color (solid = normal, dashed = shear) so the figure reads in gray.
abs_tn = np.abs(tn_vals)
ax.plot(theta_rad, abs_tn, color='steelblue', lw=2.5, linestyle='-',
        label=r'Normal traction $|\sigma_n|$ (solid)')
ax.plot(theta_rad, ts_vals, color='tomato', lw=2.5, linestyle='--',
        label=r'Shear traction $\tau_s$ (dashed)')
ax.fill_between(theta_rad, 0, abs_tn, where=(tn_vals > 0), color='steelblue', alpha=0.12,
                label=r'$\sigma_n > 0$ (tension, tinted)')
ax.fill_between(theta_rad, 0, abs_tn, where=(tn_vals < 0), facecolor='none', edgecolor='gray',
                hatch='...', lw=0, label=r'$\sigma_n < 0$ (compression, dotted)')
ax.set_rlim(0, 1.25 * max(abs_tn.max(), ts_vals.max()))

# Mark the principal stresses (the planes of zero shear)
ax.annotate(rf'$\sigma_1 = {p1_val:.1f}$', xy=(theta_p1, p1_val),
            xytext=(theta_p1 + 0.35, p1_val + 8), fontsize=11,
            arrowprops=dict(arrowstyle='->', color='black'))
ax.annotate(rf'$\sigma_2 = {p2_val:.1f}$', xy=(theta_p1 + np.pi/2, abs(p2_val)),
            xytext=(theta_p1 + np.pi/2 + 0.45, abs(p2_val) + 25), fontsize=11,
            arrowprops=dict(arrowstyle='->', color='black'))

# Mark the maximum shear stress (45 degrees from the principal axes)
theta_tau = theta_p1 + np.pi/4
ax.annotate(rf'$\tau_{{max}} = {max_ts:.1f}$', xy=(theta_tau, max_ts),
            xytext=(theta_tau + 0.35, max_ts + 22), fontsize=11,
            arrowprops=dict(arrowstyle='->', color='black'))

fig.suptitle('Stress Variation with Plane Orientation\n'
             rf'($\sigma_x={SIGMA_X:g}$, $\sigma_y={SIGMA_Y:g}$, $\tau_{{xy}}={TAU_XY:g}$ MPa)',
             fontsize=14, y=0.99)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.06), ncol=2, fontsize=10, frameon=False)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / 'fig14_nb1_traction_cauchy_polar.png', bbox_inches='tight', dpi=SAVE_DPI)
plt.show()

print(f"Principal stresses: sigma_1 = {p1_val:.2f} MPa,  sigma_2 = {p2_val:.2f} MPa")
print(f"Principal angle:    theta_p = {np.degrees(theta_p1):.2f} deg")
print(f"Max shear stress:   tau_max = {max_ts:.2f} MPa")

---
## Where to take this next

This notebook is a starting point. A few concrete extensions, each of which builds directly on the machinery above:

- **Add a Mohr's circle panel.** Plot the same state as $(\sigma_n, \tau_s)$ points in the stress plane and confirm they trace a circle centered at $(\sigma_x+\sigma_y)/2$ with radius equal to `max_ts`. This is the geometric companion to the polar sweep and previews the construction developed in Chapter 17.
- **Sweep the loading.** Wrap the whole computation in a function of `(SIGMA_X, SIGMA_Y, TAU_XY)` and animate the polar plot as $\tau_{xy}$ grows from 0. Watch the principal directions rotate away from the coordinate axes — the visual argument for why an arbitrarily oriented single gage cannot resolve a biaxial state.
- **Go to 3D.** Replace the $2\times2$ tensor with the full $3\times3$ Cauchy tensor from the chapter, sample plane normals over the unit sphere, and shade each by normal or shear traction. Setting $\sigma_z = \tau_{xz} = \tau_{yz} = 0$ recovers the free-surface (plane-stress) condition every surface-bonded gage actually experiences.